In [12]:
%pip install pyprind
%pip install pandas
%pip install numpy
%pip install scikit-learn
%pip install curl-cffi
%pip install torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 10.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 24.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 22.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Downloading the Datasets

In [1]:
from curl_cffi import requests

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
# Use 'impersonate' to mimic a real browser (e.g., chrome124)
response = requests.get(url, impersonate="chrome124")

# Check if the request was successful
if response.status_code == 200:
    with open("aclImdb_v1.tar.gz", "wb") as f:
        f.write(response.content)
    print("File downloaded successfully.")

File downloaded successfully.


In [2]:
import tarfile
with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall()

## Preprocessing the Dataset into a more convenient format

In [3]:
import pyprind
import pandas as pd
import os
import sys

basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream=sys.stdout)

data = []

for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
            
            data.append([txt, labels[l]]) 
            pbar.update()

df = pd.DataFrame(data, columns=['review', 'sentiment'])


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


## Store the datasets as CSV file

In [4]:
import numpy as np
np.random.seed(0)
df=df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={"0": "review", "1": "sentiments"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


## Transform to tf-idf

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_tfidf_features(X_train, X_test):
    tfidf = TfidfVectorizer(use_idf=True, norm='l2', smooth_idf=True,
                            strip_accents=None, lowercase=False, preprocessor=None)
    
    X_train_tfidf = tfidf.fit_transform(X_train)  # fit only on train
    X_test_tfidf  = tfidf.transform(X_test)        # transform using same vocab
    
    return X_train_tfidf, X_test_tfidf, tfidf

## Cleaning Text Data

In [9]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    
    return text
    
df['review'] = df['review'].apply(preprocessor)

## Train/Test Split Data

In [10]:
from sklearn.model_selection import train_test_split

X = df['review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [14]:
X_train_tfidf, X_test_tfidf, tfidf = get_tfidf_features(X_train, X_test)
print(X_train_tfidf.shape)  
print(X_test_tfidf.shape)   

(35000, 89509)
(15000, 89509)


## Building an FNN Classifier

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import issparse
import numpy as np

class SparseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X  # stays as sparse matrix
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

train_dataset = SparseDataset(X_train_tfidf, y_train)
test_dataset  = SparseDataset(X_test_tfidf,  y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

input_size = X_train_tfidf.shape[1]
print(f"Input size (vocab size): {input_size}")

Input size (vocab size): 89509


## Define the FNN

In [21]:
class FNN_Large(nn.Module):
    def __init__(self, input_size):
        super(FNN_Large, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

model = FNN_Large(input_size)
print(model)

FNN_Large(
  (network): Sequential(
    (0): Linear(in_features=89509, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## Training a Baseline Model and Tuning Hyperparameters

In [24]:
import time

def train_model(model, train_loader, test_loader, lr=0.001, weight_decay=1e-4, epochs=10):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            train_correct, train_total = 0, 0
            for X_batch, y_batch in train_loader:
                preds = model(X_batch).squeeze()
                train_correct += ((preds >= 0.5) == y_batch).sum().item()
                train_total   += y_batch.size(0)

            test_correct, test_total = 0, 0
            for X_batch, y_batch in test_loader:
                preds = model(X_batch).squeeze()
                test_correct += ((preds >= 0.5) == y_batch).sum().item()
                test_total   += y_batch.size(0)

        train_acc = train_correct / train_total
        test_acc  = test_correct  / test_total
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    
    elapsed = time.time() - start
    print(f"\nTime: {elapsed:.2f}s | Final Test Accuracy: {test_acc:.4f}")
    return test_acc, elapsed

In [26]:
print("=" * 50)
print("Baseline FNN: 3 hidden layers [256 → 128 → 64]")
print("=" * 50)
baseline_model = FNN_Large(input_size)
baseline_acc, baseline_time = train_model(baseline_model, train_loader, test_loader)

Baseline FNN: 3 hidden layers [256 → 128 → 64]
Epoch 1/10 | Train Acc: 0.9662 | Test Acc: 0.9016
Epoch 2/10 | Train Acc: 0.9865 | Test Acc: 0.8918
Epoch 3/10 | Train Acc: 0.9876 | Test Acc: 0.8827
Epoch 4/10 | Train Acc: 0.9869 | Test Acc: 0.8754
Epoch 5/10 | Train Acc: 0.9904 | Test Acc: 0.8789
Epoch 6/10 | Train Acc: 0.9930 | Test Acc: 0.8753
Epoch 7/10 | Train Acc: 0.9904 | Test Acc: 0.8736
Epoch 8/10 | Train Acc: 0.9931 | Test Acc: 0.8757
Epoch 9/10 | Train Acc: 0.9934 | Test Acc: 0.8749
Epoch 10/10 | Train Acc: 0.9951 | Test Acc: 0.8746

Time: 375.39s | Final Test Accuracy: 0.8746
